In [34]:
import json
import pandas as p
from base.download import collect_scores
from sga.safe import Safe
import pickle
import numpy as np
from scipy.stats import hypergeom

In [2]:
dataset = Dataset.objects.all()[1]

In [3]:
name2node = {}

for n in json.load(open(dataset.static_path('nodes.json')))['nodes']:
    name2node[n['orf'].lower()] = n['id']
    if n['name']:
        name2node[n['name'].lower()] = n['id']
    if n['alel']:
        name2node[n['alel'].lower()] = n['id']
    for a in n['aliases']:
        if a and a.lower() not in name2node:
            name2node[a.lower()] = n['id']

In [23]:
xls = p.read_excel('/home/matej/conditions_nodiff_filter_all.xlsx', sheetname='Selection 0.95')
orfs = list(xls.orf.unique())

In [28]:
# ORF independent requirements
with open(dataset.static_path('nodes_inv.pickle'), 'rb') as fp:
    nodes_inv = pickle.load(fp)

annotation = dataset.default_annotation

nodes_inv_inv = {}
for nid, sids in nodes_inv.items():
    for sid in sids:
        nodes_inv_inv[sid] = nid

terms = {}
node_in_terms = {}
all_annotated = 0
for term in Term.objects.filter(annotation=annotation).prefetch_related('strains', 'genes__strain_set'):
    term_nodes = set()

    if annotation.version == Annotation.VERSION_STRAINS:
        for strain in term.strains.all():
            if strain.pk in nodes_inv_inv:
                term_nodes.add(nodes_inv_inv[strain.pk])
    else:
        for gene in term.genes.all():
            for strain in gene.strain_set.all():
                if strain.pk in nodes_inv_inv:
                    term_nodes.add(nodes_inv_inv[strain.pk])

    terms[term] = term_nodes
    all_annotated += len(term_nodes)

    for n in term_nodes:
        node_in_terms.setdefault(n, []).append(term)

node_map = {}
for n in json.load(open(dataset.static_path('nodes.json')))['nodes']:
    node_map[n['id']] = n

In [43]:
results = {}

i = len(orfs)
for o in orfs:
    i -= 1
    
    print(i, o)
    if o.lower() not in name2node:
        results[o] = 'Not on the map'
        continue
    
    node = name2node[o.lower()]
    data = collect_scores(dataset, [node])
    
    attributes = []
    for s, t, w in data.itertuples(index=False):
        row = [None, 0]

        if w < 0:
            row[1] = 1
        else:
            continue

        if int(s) == node:
            row[0] = t
        else:
            row[0] = s

        attributes.append(row)
    
    if not attributes:
        results[o] = 'No negative interactions'
        continue
    
    attributes = p.DataFrame(attributes, columns=['node', 'negatives']).set_index('node')
    safe = Safe(dataset.static_path('safe_layout.csv'), attributes, dataset.static_path('safe_neighbors.csv'))
    safe.prepare_attributes()

    enrichments = safe.calculate()
    
    enrichments = enrichments.loc[enrichments.any(axis=1)]

    gene_lists = []

    Fj = safe.attributes.sum().to_dict()  # Number of nodes in the network given any attribute
    M1 = all_annotated
    
    col = 'negatives'
    attr_nodes = safe.attributes[col]
    attr_nodes = set(attr_nodes[attr_nodes.astype(bool)].index)
    enr_nodes = set((enrichments[col][enrichments[col] > -np.log10(0.05 / len(Fj)) / 16.0]).index)
    n1 = len(enr_nodes.intersection(attr_nodes))
    data2 = []

    for term, term_nodes in terms.items():
        k1 = len(enr_nodes.intersection(term_nodes).intersection(attr_nodes))
        N1 = len(term_nodes.intersection(attr_nodes))

        if k1:
            fold1 = (k1 * all_annotated) / float(n1 * len(term_nodes))  # N1

            gene_lists.append(((col, term, enr_nodes.intersection(term_nodes).intersection(attr_nodes))))
            # ','.join([node_map[n]['label'] for n in enr_nodes.intersection(term_nodes)
            # .intersection(attr_nodes)])

            data2.append((
                term.alias,
                hypergeom.pmf(k1, M1, n1, N1),
                fold1,
                '%d / %d, %.1f%%' % (n1, len(attr_nodes), n1 * 100. / len(attr_nodes)),
                '%d / %d, %.1f%%' % (k1, n1, k1 * 100. / n1),
                '%d / %d, %.1f%%' % (len(term_nodes), all_annotated, len(term_nodes) * 100. / all_annotated),
            ))

    colnames = ['Term', 'p-value', 'fold change',
                'Fraction of input gene list annotated to a bioprocess cluster', 'Cluster frequency',
                'Background frequency']

    res_data = p.DataFrame(data2, columns=colnames).sort_values('p-value')
    results[o] = res_data

507 YJL207C
506 YDR084C
505 YIL158W
504 YNR030W
503 YDR122W
502 YPR194C
501 YBR130C
500 YMR155W
499 YMR221C
498 YLR327C
497 YFR016C
496 YKL015W
495 YGL134W
494 YMR101C
493 YPL111W
492 YPR011C
491 YPR003C
490 YBR010W
489 YER180C-A
488 YLR303W
487 YNR022C
486 YPL107W
485 YBL085W
484 YFL055W
483 YPL199C
482 YHR210C
481 YLR156W
480 YNL031C
479 YHR143W
478 YPL056C
477 YGR212W
476 YJR061W
475 YOL111C
474 YMR316W
473 YMR171C
472 YAL064W
471 YHL030W
470 YMR152W
469 YGL117W
468 YLR090W
467 YGR170W
466 YAL017W
465 YOR034C
464 YGR102C
463 YLL012W
462 YBR139W
461 YER134C
460 YIL164C
459 YAL029C
458 YOL013C
457 YLR404W
456 YNL011C
455 YIR028W
454 YBL061C
453 YGL094C
452 YMR273C
451 YOR016C
450 YDR216W
449 YKL120W
448 YHR003C
447 YLR313C
446 YBR255C-A
445 YDR414C
444 YDR524W-C
443 YBR137W
442 YAL014C
441 YOL159C
440 YGR233C
439 YLR390W-A
438 YDR357C
437 YBR233W
436 YBL005W
435 YKR080W
434 YBR092C
433 YBR294W
432 YDL091C
431 YDR318W
430 YJR140C
429 YAL023C
428 YDR242W
427 YDR090C
426 YDL159W-A
425 YD

In [49]:
numenriched = []
for orf, enr in results.items():
    if isinstance(enr, str):
        numenriched.append([orf, enr])
    else:
        numenriched.append([orf, enr.shape[0]])
numenriched = p.DataFrame(numenriched, columns=['ORF', '# enriched'])

In [56]:
enrichment_list = []
for orf, enr in results.items():
    if isinstance(enr, str):
        continue
    
    if 0 in enr.shape:
        continue
    
    tmp = enr.copy()
    tmp.loc[:,'ORF'] = orf
    enrichment_list.append(tmp)
enrichment_list = p.concat(enrichment_list)

In [58]:
xlout = p.ExcelWriter('/home/matej/conditional_enrichments_safe.xlsx')
numenriched.to_excel(xlout, sheet_name='Summary', index=False)
enrichment_list.to_excel(xlout, sheet_name='Enrichments', index=False)
xlout.save()